# Week 21 Label Studio follow-up export

This notebook records the reproducible steps for the pattern-positive follow-up export. It first syncs local Label Studio annotations into CSV tracking files, then exports only data-source/sort-variable groups where at least one previous image was labeled as a pattern class, with no repeated `dataset_key + channel_name + sort_variable` combinations.

In [ ]:
from pathlib import Path
import os
import subprocess

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / 'notebooks' / 'week_21').exists():
    REPO_ROOT = Path.cwd().parents[1]
WEEK21 = REPO_ROOT / 'notebooks' / 'week_21'
EXPORT_ROOT = WEEK21 / 'labelstudio_export_pattern_positive_followup_1000'
REPO_ROOT

## 1. Sync Label Studio annotations into CSV files

In [ ]:
subprocess.run([
    'python3',
    str(WEEK21 / 'update_labelstudio_annotation_tracking.py'),
], cwd=REPO_ROOT, check=True)

## 2. Export follow-up images

The Julia script reuses the Week-20 model predictions for ranking, but filters candidates to sort variables with at least one previous non-`no_class` label. The reference fixation dataset is handled separately from its local HDF5/events files.

In [ ]:
subprocess.run([
    'julia',
    '--project=notebooks/model_test',
    str(WEEK21 / 'export_pattern_positive_followup_1000.jl'),
], cwd=REPO_ROOT, check=True)

## 3. Import into local Label Studio

This uses the same local SQLite/Django import route as the previous Week-21 import. It assumes Label Studio was started with local-file serving enabled and `notebooks/week_21` as the document root.

In [ ]:
env = os.environ.copy()
env.update({
    'DEBUG': 'false',
    'LATEST_VERSION_CHECK': 'false',
    'LABEL_STUDIO_BASE_DATA_DIR': str(REPO_ROOT / '.label-studio-data'),
    'LOCAL_FILES_SERVING_ENABLED': 'true',
    'LOCAL_FILES_DOCUMENT_ROOT': str(WEEK21),
    'LABEL_STUDIO_LOCAL_FILES_SERVING_ENABLED': 'true',
    'LABEL_STUDIO_LOCAL_FILES_DOCUMENT_ROOT': str(WEEK21),
    'DJANGO_SETTINGS_MODULE': 'label_studio.core.settings.label_studio',
})
ld_path = REPO_ROOT / '.tools' / 'python312' / 'usr' / 'lib'
if ld_path.exists():
    env['LD_LIBRARY_PATH'] = f"{ld_path}:{env.get('LD_LIBRARY_PATH', '')}"
env['PYTHONPATH'] = str(REPO_ROOT / '.venvs' / 'labelstudio' / 'lib' / 'python3.12' / 'site-packages' / 'label_studio')

subprocess.run([
    str(REPO_ROOT / '.venvs' / 'labelstudio' / 'bin' / 'python'),
    str(WEEK21 / 'import_pattern_positive_followup_labelstudio.py'),
], cwd=REPO_ROOT, env=env, check=True)

In [ ]:
import csv

with (EXPORT_ROOT / 'summary.csv').open(newline='') as f:
    summary = list(csv.DictReader(f))
[(r['dataset_key'], r['exported_count'], r['positive_sort_variables']) for r in summary if int(r['exported_count']) > 0]